In [ ]:
from jedi.inference.compiled import value

#Q3 stroke 与 price 的相关系数
df[['stroke','price']].corr()

#Q3是否线性关系？用 regplot() 验证
sns.regplot(x='stroke', y='price', data=df)
plt.ylim(0,)
plt.show()

#必须"数值 + 图形"双证据：先由 |r|≈0.08 作出判断，再用 regplot 佐证散点极度分散、回归线近乎水平。

# 数值 vs 数值	regplot	engine-size vs price
# 分类 vs 数值	boxplot	本格：body-style vs price
#画一张箱线图，用来研究分类变量 body-style（车身样式）对数值变量 price（价格）的影响——这是 notebook "Categorical Variables" 小节的第一张图。
#和 regplot 的关键区别：x 这里放的是类别而不是连续数值，所以不再是一条回归线，而是每个类别一根箱子。
    #   ┌─┐ ← 最大值（须线外为离群点 ○）
    #   │ │
    # ┌─┴─┴─┐ ← 上四分位数 Q3 (75%)
    # │     │
    # ├─────┤ ← 中位数 Median (50%)，箱内那条线
    # │     │
    # └─┬─┬─┘ ← 下四分位数 Q1 (25%)
    #   │ │
    #   └─┘ ← 最小值
# 不同类别的箱子重叠严重 → 类别对价格区分度低 → 差的预测变量（如 body-style）；
# 不同类别的箱子几乎不重叠 → 区分度高 → 好的预测变量（如 engine-location、drive-wheels）。
#一句话总结：这格代码把车按 5 种车身样式分组，各画一个价格分布箱——结果箱子大量重叠，说明“车身样式”这个分类特征对预测车价帮助不大。


#这就是箱线图读图的通用准则：分离 = 有区分度 = 有预测价值；重叠 = 区分不了 = 没用。
#  5 个车身样式的箱子高度重叠严重（尤其是 sedan、hatchback、wagon 的价格区间几乎叠在一起，中位数也差不多）→ 说明知道车身样式，并不能很好地推断价格；
sns.boxplot(x="body-style", y="price", data=df)

# 含义：只要知道引擎在哪，就能大致区分这车是贵还是便宜——后置引擎的车（如保时捷 911 这类跑车）普遍贵，前置的普通家用车便宜；
sns.boxplot(x="engine-location", y="price", data=df)

sns.boxplot(x="engine-location", y="price", data=df)

# 图	                     箱子关系	            样本量	                    结论
# body-style vs price	        大量重叠	        均衡	                  ✗ 不适合
# engine-location vs price	     明显分离	    严重不均（rear 仅 3）	   ⚠️ 不可下结论
# drive-wheels vs price（本格）	  明显不同	      较均衡	                    ✓ 好预测变量

# 一句话总结：本格代码画三种驱动方式的价格箱线图，发现后驱车明显更贵且样本量充足——drive-wheels 是本数据集中少数被确认“靠谱”的分类特征，后面第 4 节的 groupby 平均价将用数字再次验证它。


# 写法	统计哪些列	行为
# df.describe()	只统计数值列（int64/float64），object 列直接被丢弃	默认行为
# df.describe(include=['object'])	只统计 object 列	数值列被排除
# df.describe(include='all')	数值列 + object 列全部要	一个都不落下
df.describe()
df.describe(include=['object'])
#count	非空值的数量 unique是该列有多少个不同的类别（去重后的取值个数） top	出现次数最多的类别（众数） freq	众数出现了多少次
#              make  fuel-type  aspiration  num-of-doors  body-style  drive-wheels  engine-location
# count         201        201         201           201         201           201              201
# unique         22          2           2             2           5             3                2
# top          toyota        gas         std          four      sedan           rwd            front
# freq           31        180         159           114          94           76              198
#所以带有include=object，只能统计这些内容

#统计类型个数
df['drive-wheels'].value_counts()
# Series 转换为 DataFrame（单列表格），方便后续重命名列、展示或继续操作
df['drive-wheels'].value_counts().to_frame()

drive_wheels_counts = df['drive-wheels'].value_counts().to_frame()
drive_wheels_counts.rename(columns={'drive-wheels': 'value_counts'}, inplace=True)
drive_wheels_counts

drive_wheels_counts.index.name = 'drive-wheels'
drive_wheels_counts

engine_loc_counts = df['engine-location'].value_counts().to_frame()
engine_loc_counts.rename(columns={'engine-location': 'value_counts'}, inplace=True)
engine_loc_counts.index.name = 'engine-location'
engine_loc_counts.head(10)

df['drive-wheels'].unique()

df_group_one = df[['drive-wheels','body-style','price']]

#as_index=False	让分组键保留为普通列，而不是变成行索引
df_group_one = df_group_one.groupby(['drive-wheels'],as_index=False).mean()
df_group_one

# drive-wheels    body-style      price

# 4wd           hatchback  10311.764706
# fwd           hatchback   9409.006897
# rwd           sedan      18519.700000


#     drive-wheels  body-style        price
# 0          4wd  hatchback  10311.764706
# 1          fwd  hatchback   9409.006897
# 2          rwd  sedan      18519.700000

df_gptest = df[['drive-wheels','body-style','price']]
grouped_test1 = df_gptest.groupby(['drive-wheels','body-style'],as_index=False).mean()
grouped_test1

#作用是把分组均值结果重排成透视表，为后面的热力图可视化做准备
# index='drive-wheels'	该列的每个唯一值变成透视表的行（rwd / fwd / 4wd 共 3 行）
#columns='body-style'	该列的每个唯一值变成透视表的列（convertible / hatchback / sedan / wagon / hardtop 共 5 列）
#取两列，一列变行，一列变列
# values（未指定）	—	自动对剩余的数值列（只有 price）取值填入单元格;默认这里只有一行，所以只取price
grouped_pivot = grouped_test1.pivot(index='drive-wheels',columns='body-style')
grouped_pivot

#               price
# body-style    convertible     hardtop    hatchback        sedan       wagon
# drive-wheels
# 4wd                NaN          NaN  10311.764706  12647.263158  NaN
# fwd                NaN   11894.5000   9409.006897   9811.800000  NaN
# rwd         23633.000000  22143.9250  10363.440000  24543.666667  NaN


#缺失数据本身是个复杂问题，这里填 0 只是为了让后面的热力图能画出来。这里是将Nan填为0
grouped_pivot = grouped_pivot.fillna(0) #fill missing values with 0
grouped_pivot
#pivot 本身不产生任何新信息，它纯粹是为了可视化服务的重排
# 长表适合给 pandas 读，宽表适合给人看和给 plt.pcolor()（第 323、331 行）画热力图
# 这两行代码把"按驱动方式 × 车身样式分组的平均价格"长表转置成行=drive-wheels、列=body-style 的二维透视表（类似 Excel 交叉表），暴露出组合缺失的 NaN 单元格，为下一步 fillna(0) 和热力图绘制铺路


